# Assignment 1 — Data Preparation

## 1. Identifying the Prediction Target

The target is y - whether the client subscribed to a term deposit. It's the right column to predict because it's the actual outcome of the campaign and isn't known until after the interaction is complete.

Two columns that look like valid targets but aren't:
- duration - call duration in seconds. Only known after the call ends, so it's unavailable at prediction time. Using it would be data leakage — the model would be cheating.
- campaign - number of contacts made during this campaign. This is an input to the process controlled by the bank, not the client's response. It describes what was done, not what happened as a result.

## 2. Data Loading and Exploration

In [ ]:
# standard data manipulation libraries
import pandas as pd
import numpy as np

# plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# splitting the dataset into train/val/test
from sklearn.model_selection import train_test_split

# preprocessing: scaling numbers, encoding categories
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

# the model we're using
from sklearn.linear_model import LogisticRegression

# metrics to evaluate how well the model did
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

# for dealing with class imbalance (more no's than yes's)
from imblearn.over_sampling import SMOTE

In [ ]:
# load the dataset - sep=';' because this csv uses semicolons not commas
df = pd.read_csv('bank-additional-full.csv', sep=';')

# quick sanity check: how many rows and columns do we have
print(f"Shape: {df.shape}  ({df.shape[0]} rows, {df.shape[1]} columns)")

# print all column names so we know what we're working with
print(f"\nColumn names:\n{list(df.columns)}")

In [ ]:
# shows column names, data types, and how many non-null values each column has
# useful to spot missing values and check if types look right (e.g. numbers stored as strings)
df.info()

In [ ]:
# preview the first 5 rows to get a feel for what the data looks like
df.head()

In [ ]:
# summary stats for all numerical columns: mean, std, min, max, quartiles
# helps spot things like extreme values or weird ranges (e.g. pdays maxing at 999)
df.describe()

In [ ]:
# select_dtypes filters columns by data type
# include='number' gets all numeric columns (int and float)
numerical_cols = df.select_dtypes(include='number').columns.tolist()

# include='object' gets all text/string columns
categorical_cols = df.select_dtypes(include='object').columns.tolist()

print(f"Numerical ({len(numerical_cols)}):   {numerical_cols}")
print(f"Categorical ({len(categorical_cols)}): {categorical_cols}")

In [ ]:
# explicit missing values - pandas marks these as NaN
# isnull().sum() counts how many NaN values per column
print("=== Explicit NaN values ===")
print(df.isnull().sum())

# implicit missing values - this dataset uses the string "unknown" instead of NaN
# (df == 'unknown') gives a True/False table, .sum() counts Trues per column
print("\n=== Implicit missing values ('unknown') ===")
print((df == 'unknown').sum())

No actual NaN values, but several columns use `"unknown"` as a placeholder — `default` is the worst (21% unknown). Also, `pdays = 999` means the client was never previously contacted, not an actual day count.

In [ ]:
# age distribution - want to see the spread of client ages
plt.figure(figsize=(8, 4))
sns.histplot(df['age'], bins=30, kde=True)  # kde=True adds a smooth density curve on top
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()

In [ ]:
# call duration distribution - this is very skewed (most calls are short)
# also note: we'll drop this column before modeling because
# you only know the duration after the call ends, so it can't be used for prediction
plt.figure(figsize=(8, 4))
sns.histplot(df['duration'], bins=50, kde=True)
plt.title('Call Duration Distribution (seconds)')
plt.xlabel('Duration')
plt.ylabel('Count')
plt.show()

In [ ]:
# horizontal bar chart of job types, sorted by frequency
# order= sorts bars from most to least common
plt.figure(figsize=(10, 5))
sns.countplot(y='job', data=df, order=df['job'].value_counts().index)
plt.title('Job Type Distribution')
plt.xlabel('Count')
plt.show()

In [ ]:
# count how many yes/no in the target column
counts = df['y'].value_counts()

# also get percentages (normalize=True gives proportions, x100 converts to %)
pcts = df['y'].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(counts.index, counts.values, color=['steelblue', 'salmon'])

# add percentage labels above each bar
for bar, pct in zip(bars, pcts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{pct:.1f}%', ha='center', fontsize=12)

ax.set_title('Target Variable Distribution')
ax.set_xlabel('Subscribed (y)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

- **Age** — roughly normal, most clients between 25–55.
- **Duration** — very skewed, and more importantly not available at prediction time (you only know the call length after it ends). Will drop this before modeling.
- **Job** — admin, blue-collar, and technician dominate. There's an "unknown" category too.
- **Target (y)** — 89% "no", 11% "yes". A model that always predicts "no" gets 89% accuracy, so accuracy alone is a bad metric here.

---
## 3. Task Ordering

The tasks are listed alphabetically in the assignment, but the order matters a lot. The rule is: **split before fitting any transformation**. Once you split, all learned parameters (encoder categories, scaler mean/std, imputation values, SMOTE) must come from training only.

| Step | Task | What's allowed | What must NOT be used | Leakage if wrong |
|------|------|---------------|----------------------|-----------------|
| 1 | Identify target | All columns, raw data | Nothing to fit yet | N/A |
| 2 | Explore data | Full dataset (just reading) | Can't fit transformations | N/A — we're only looking |
| 3 | Task ordering | Knowledge of the problem | — | N/A |
| 4 | **Split data** | Full dataset | — | **Boundary** — everything below is training-only |
| 5 | Missing values | Training set only for mode/imputation | Val/test statistics | Imputing with global mean leaks test distribution |
| 6 | Encode categoricals | Training categories only | Test set categories | Encoder that "knows" test categories inflates performance |
| 7 | Scale features | Training mean/std only | Val/test mean/std | Scaler fitted on all data leaks test stats into normalization |
| 8 | Feature selection | Training correlations/variance | Test set patterns | Removing features based on full-data correlation uses test info |
| 9 | SMOTE | Training set only | Val/test samples | Synthetic samples from test neighbors contaminate training |
| 10 | Train model | Resampled training set | Val/test labels | Evaluating on data seen during training isn't a real test |

**Example of bad ordering:** say you scale first, then split. The scaler computes mean and std from all 41k rows including the test set. When the model is evaluated on test, those samples already influenced the preprocessing — the test set isn't truly unseen. Performance looks better than it would be in production.

---
## 4. Data Splitting

Split before any learned transformations. Dropping `duration` first (data leakage), then a 60/20/20 stratified split. Stratification ensures the same 11% positive rate in all three sets — without it, you could end up with almost no "yes" samples in one split by chance.

In [ ]:
# drop duration - can't know this before the call ends, so it would be data leakage
df = df.drop(columns=['duration'])

# convert target from strings to numbers: 'no' -> 0, 'yes' -> 1
df['y'] = df['y'].map({'no': 0, 'yes': 1})

# separate features (X) from the target (y)
X = df.drop(columns=['y'])
y = df['y']

# first split: 60% train, 40% temp
# stratify=y keeps the same yes/no ratio across all splits
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

# second split: cut the 40% in half -> 20% validation, 20% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# check sizes and that the positive rate (~11%) is preserved in each split
print(f"Train: {X_train.shape[0]} samples ({y_train.mean():.3f} positive rate)")
print(f"Val:   {X_val.shape[0]} samples ({y_val.mean():.3f} positive rate)")
print(f"Test:  {X_test.shape[0]} samples ({y_test.mean():.3f} positive rate)")

Positive rate is ~11.3% across all three sets — stratification worked.

---
## 5. Managing Missing Values

No real NaNs, but two forms of hidden missingness:

**Data cleaning decisions** (fixing bad data entries):
- `"unknown"` strings in categorical columns — for columns with few unknowns (job 0.8%, marital 0.2%, education 4.3%, housing/loan 2.5%) I replace with the mode from the training set. These are likely just missing data, not meaningful.
- `pdays = 999` — this is a sentinel code meaning "never previously contacted", not an actual day count. I replace it with 0 and create a separate binary flag.

**Modeling decisions** (cases where missingness itself might carry information):
- `default = "unknown"` — 21% of training rows have this. That's too much to impute without introducing serious noise. More importantly, a client with unknown credit default status might behave differently from one with a known status — so keeping `"unknown"` as its own category lets the model learn from it.

Mode is computed from training only, then applied to val/test. Otherwise the imputation would use statistics from data the model shouldn't have seen.

In [ ]:
# check how many 'unknown' values are in each categorical column of the training set
# only look at training from this point on - never peek at val/test stats
print("Unknown counts in training set:")
for col in X_train.select_dtypes(include='object').columns:
    n_unknown = (X_train[col] == 'unknown').sum()
    if n_unknown > 0:
        print(f"  {col}: {n_unknown} ({n_unknown/len(X_train)*100:.1f}%)")

# pdays = 999 is a sentinel meaning "never previously contacted" - not a real day count
print(f"\npdays == 999 in training set: {(X_train['pdays'] == 999).sum()} ({(X_train['pdays'] == 999).mean()*100:.1f}%)")

In [ ]:
# columns to impute - we leave 'default' alone because 21% unknown is too many to impute reliably
cols_to_impute = ['job', 'marital', 'education', 'housing', 'loan']

for col in cols_to_impute:
    # find the most common value in training (excluding 'unknown' rows)
    mode_val = X_train[col][X_train[col] != 'unknown'].mode()[0]
    print(f"  {col}: replacing 'unknown' with '{mode_val}'")
    
    # apply the same replacement to all three splits using the training mode
    # never recompute the mode from val or test
    X_train[col] = X_train[col].replace('unknown', mode_val)
    X_val[col] = X_val[col].replace('unknown', mode_val)
    X_test[col] = X_test[col].replace('unknown', mode_val)

# handle pdays = 999 sentinel
for dataset in [X_train, X_val, X_test]:
    # create a binary flag: 1 if the client was previously contacted, 0 if not
    dataset['previously_contacted'] = (dataset['pdays'] != 999).astype(int)
    # replace 999 with 0 so the number makes sense (0 days ago if never contacted)
    dataset['pdays'] = dataset['pdays'].replace(999, 0)

print(f"\nUnknowns left in training set: {(X_train == 'unknown').sum().sum()}")
print(f"(all in 'default' column - keeping those on purpose)")

---
## 6. Encoding Categorical Variables

Logistic regression needs numbers, so the categorical columns need encoding.

- **`education`** → ordinal encoding, since there's a clear order (illiterate < basic.4y < ... < university.degree). The model can learn one coefficient that captures the directional effect of education level.
- **Everything else** → one-hot encoding, since there's no meaningful order (e.g. job types, months). Using `drop='first'` avoids perfect multicollinearity — the dropped category becomes the reference point.

Both encoders are fit on training only. If we fitted on all data, the encoder would know about categories in the test set before seeing them.

**Effect on the dataset:**
- *Dimensionality*: went from 10 categorical columns to 33 encoded ones (32 OHE + 1 ordinal). Each category gets its own binary column, so the model can learn a separate coefficient for each job type, month, etc.
- *Interpretability*: each OHE coefficient now represents the effect of being in that specific category vs the reference category. This is more interpretable than, say, label encoding where the model would assume ordinal relationships that don't exist.
- *Decision boundaries*: logistic regression draws a linear boundary in feature space. OHE lets the model shift that boundary differently for each category value — something it couldn't do if we just assigned arbitrary integers to categories.

In [ ]:
# check which columns are still text after imputation - these all need encoding
cat_cols = X_train.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns: {cat_cols}")
print(f"Before encoding - shape: {X_train.shape}")

In [ ]:
# education has a real order (illiterate < basic.4y < ... < university.degree)
# so we use ordinal encoding - assigns a number based on position in this list
edu_order = ['illiterate', 'basic.4y', 'basic.6y', 'basic.9y', 
             'high.school', 'university.degree', 'professional.course']

# handle_unknown='use_encoded_value' + unknown_value=-1 means unseen categories get -1
ord_enc = OrdinalEncoder(categories=[edu_order], handle_unknown='use_encoded_value', unknown_value=-1)

# fit on training only, then apply the same mapping to val and test
X_train['education'] = ord_enc.fit_transform(X_train[['education']])
X_val['education'] = ord_enc.transform(X_val[['education']])
X_test['education'] = ord_enc.transform(X_test[['education']])

print("Education encoding:")
for i, level in enumerate(edu_order):
    print(f"  {level} -> {i}")

In [ ]:
# all remaining categorical columns (education already handled above)
nominal_cols = [c for c in cat_cols if c != 'education']
print(f"One-hot encoding these: {nominal_cols}")

# sparse_output=False - return a regular array instead of a sparse matrix
# handle_unknown='ignore' - if val/test has unseen category, just set all its columns to 0
# drop='first' - remove one column per feature to avoid perfect multicollinearity
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')

# fit on training only
ohe.fit(X_train[nominal_cols])

# transform each split and wrap in a DataFrame to preserve column names and row indices
train_ohe = pd.DataFrame(
    ohe.transform(X_train[nominal_cols]),
    columns=ohe.get_feature_names_out(nominal_cols),
    index=X_train.index
)
val_ohe = pd.DataFrame(
    ohe.transform(X_val[nominal_cols]),
    columns=ohe.get_feature_names_out(nominal_cols),
    index=X_val.index
)
test_ohe = pd.DataFrame(
    ohe.transform(X_test[nominal_cols]),
    columns=ohe.get_feature_names_out(nominal_cols),
    index=X_test.index
)

# drop the original text columns and attach the new encoded columns
X_train = X_train.drop(columns=nominal_cols).join(train_ohe)
X_val = X_val.drop(columns=nominal_cols).join(val_ohe)
X_test = X_test.drop(columns=nominal_cols).join(test_ohe)

print(f"\nAfter encoding - shape: {X_train.shape}")
print(f"Went from {len(cat_cols)} categorical columns to {len(ohe.get_feature_names_out(nominal_cols))} one-hot + 1 ordinal")

---
## 7. Feature Scaling

Logistic regression uses gradient descent, which is sensitive to feature scale. If one feature spans 0–5000 and another 0–1, gradient steps are dominated by the large-scale feature and convergence is slow.

I'm using **StandardScaler** (subtract mean, divide by std) because:
- Gradient descent converges faster and more stably when features are on the same scale
- Coefficients become directly comparable — a larger coefficient genuinely means that feature has more influence, not just that it was measured on a bigger scale
- Regularization (L2 penalty by default in sklearn's LogisticRegression) penalizes large coefficients equally across all features. Without scaling, features with large raw values would get penalized more heavily just because of their units, which makes no sense

Fit on training only, then applied to val and test using the same training mean/std.

In [ ]:
# list the numerical columns that need scaling
# OHE columns are already 0/1 so they don't need it
num_cols = ['age', 'education', 'campaign', 'pdays', 'previous', 
            'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 
            'euribor3m', 'nr.employed', 'previously_contacted']

print("Before scaling (training set):")
print(X_train[num_cols].describe().round(2).loc[['mean', 'std']])

# StandardScaler: subtracts the mean and divides by std so each feature is centered at 0
# fit_transform on training: learns mean/std from training data, then scales it
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

# transform only on val/test: reuses the training mean/std - do NOT refit here
X_val[num_cols] = scaler.transform(X_val[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print("\nAfter scaling (training set):")
print(X_train[num_cols].describe().round(2).loc[['mean', 'std']])

## 8. Feature Selection

Two things to check, both on training data only:
1. **Low variance** — features that barely change can't help the model. Threshold: 0.01.
2. **High correlation** — correlated features are redundant and cause unstable coefficients in logistic regression. Threshold: 0.8.

In [ ]:
from sklearn.feature_selection import VarianceThreshold

# --- check 1: low variance ---
# features that barely change are useless - the model can't learn from a constant column
# threshold=0.01 means: flag features with variance below 0.01
var_check = VarianceThreshold(threshold=0.01)
var_check.fit(X_train)  # fit on training only

# get_support() returns True/False per feature - False = low variance = should drop
low_var_cols = X_train.columns[~var_check.get_support()].tolist()
print(f"Low variance features (threshold=0.01): {low_var_cols}")

# --- check 2: high correlation ---
# if two features are very correlated, one is basically redundant
# keeping both causes unstable coefficients in logistic regression
corr = X_train[num_cols].corr().abs()

# upper triangle avoids listing each pair twice (A-B and B-A)
upper_tri = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

# collect all pairs with correlation above 0.8
high_corr_pairs = []
for col in upper_tri.columns:
    for row in upper_tri.index:
        if upper_tri.loc[row, col] > 0.8:
            high_corr_pairs.append((row, col, round(upper_tri.loc[row, col], 3)))

print(f"\nHighly correlated pairs (>0.8):")
for r, c, v in high_corr_pairs:
    print(f"  {r} <-> {c}: {v}")

In [ ]:
# combine the lists: low variance cols + euribor3m
# euribor3m correlates >0.9 with both emp.var.rate and nr.employed - keeping all three causes multicollinearity
cols_to_drop = low_var_cols + ['euribor3m']

# remove duplicates and only drop cols that actually exist
cols_to_drop = list(set(cols_to_drop))
cols_to_drop = [c for c in cols_to_drop if c in X_train.columns]

print(f"Dropping: {cols_to_drop}")

# drop from all three splits
X_train = X_train.drop(columns=cols_to_drop)
X_val = X_val.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

print(f"Shape after feature selection: {X_train.shape}")

Dropped `euribor3m` (corr > 0.9 with both `emp.var.rate` and `nr.employed` — keeping all three would cause multicollinearity, making coefficients unstable and hard to interpret). Dropped `default_yes` and `month_dec` (near-constant, useless for classification). Kept both `pdays` and `previously_contacted` despite their correlation — binary flag says "was contacted before?", `pdays` says "how long ago?", slightly different info.

Feature selection is done on training data only. If we computed correlations and variance on the full dataset before splitting, we'd be using test set patterns to decide which features to keep — that's another form of data leakage. The selected feature set would be "optimized" for the test set, so evaluation would overestimate real-world performance.

---
## 9. Addressing Class Imbalance

The training set is 88.7% "no", 11.3% "yes" — roughly 8:1 imbalance. This matters because:
- **Accuracy** becomes misleading — always predicting "no" gives 88.7% accuracy with zero predictive value
- **Precision** (of predicted "yes", how many are real) gets inflated if the model rarely predicts "yes"
- **Recall** (of all real "yes", how many we catch) collapses to 0 if the model never predicts "yes"

I'm using **SMOTE** which generates synthetic minority samples by interpolating between real ones in feature space — better than just duplicating because it adds variety rather than overfitting to existing points. Using `sampling_strategy=0.5` (2:1 ratio) rather than full 1:1 balance — full balancing made the model too aggressive about predicting "yes", hurting precision significantly. The 2:1 ratio is a better tradeoff between recall and precision.

SMOTE is applied to the training set only. Val and test keep the original distribution so evaluation reflects real-world conditions. If SMOTE were applied before splitting, synthetic samples interpolated from test-set neighbors could appear in training — the model would effectively be trained on a modified version of the test set.

In [ ]:
# check how imbalanced the training set is before SMOTE
print("Training set class distribution (before SMOTE):")
print(y_train.value_counts())
print(f"Ratio: {y_train.value_counts()[0] / y_train.value_counts()[1]:.1f} : 1")

# SMOTE (Synthetic Minority Oversampling Technique) generates new fake "yes" samples
# by interpolating between existing ones in feature space
# sampling_strategy=0.5 means: make minority 50% the size of majority (2:1 ratio)
# we use 0.5 instead of 1.0 because full 1:1 balance hurt precision a lot
smote = SMOTE(sampling_strategy=0.5, random_state=42)

# only apply to training - val/test keep the original real-world distribution
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE (sampling_strategy=0.5):")
print(pd.Series(y_train_res).value_counts())
print(f"Training set went from {len(X_train)} to {len(X_train_res)} samples")

## 10. Training a Logistic Regression Model

In [ ]:
# train logistic regression on the SMOTE-resampled training set
# max_iter=1000 gives the solver enough iterations to converge (default 100 can fail)
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_res, y_train_res)

# predict on the validation set (not test - that's saved for final evaluation)
y_pred = model.predict(X_val)

# calculate evaluation metrics
acc   = accuracy_score(y_val, y_pred)
prec  = precision_score(y_val, y_pred)   # of all predicted yes, how many were actually yes
rec   = recall_score(y_val, y_pred)      # of all actual yes, how many did we correctly find
f1    = f1_score(y_val, y_pred)          # harmonic mean of precision and recall

print("=== Validation Set Results ===")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")

# zero rule baseline: a dumb model that always predicts the majority class (no)
# any useful model should beat this
baseline_acc = 1 - y_val.mean()                             # always predicts 0 -> ~88.7% accuracy
baseline_f1  = f1_score(y_val, np.zeros(len(y_val)))        # always predicts 0 -> F1 = 0
print(f"\nZero Rule baseline - Accuracy: {baseline_acc:.4f} | F1: {baseline_f1:.4f}")
print(f"Our model          - Accuracy: {acc:.4f} | F1: {f1:.4f}")

In [ ]:
# confusion matrix - shows breakdown of correct vs wrong predictions
# rows = actual class, columns = predicted class
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_val, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No', 'Yes'])
disp.plot(ax=ax, cmap='Blues', values_format='d')  # values_format='d' shows whole numbers
plt.title('Confusion Matrix - Validation Set')
plt.show()

# print each cell explicitly so it's easy to reference
print(f"\nTrue Negatives:  {cm[0][0]}")  # correctly predicted no
print(f"False Positives: {cm[0][1]}")   # predicted yes, actually no (false alarm)
print(f"False Negatives: {cm[1][0]}")   # predicted no, actually yes (missed a subscriber)
print(f"True Positives:  {cm[1][1]}")   # correctly predicted yes

Accuracy (87.3%) is close to the Zero Rule baseline (88.7%), but that's expected — we want the model to predict "yes" sometimes, not just always say "no".

The better metric here is F1. The baseline's F1 is 0.0 (it never predicts "yes" so recall is zero). Our model gets F1 ≈ 0.50, catching about 56% of actual subscribers with 45% precision. In a marketing context that's a useful result — the model finds real leads rather than calling everyone.